# PART 1

## Imports, Verifications and Stuff

In [ ]:
import torch, random, numpy as np

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print(f"Device : {DEVICE}")
print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"Seed   : {SEED}")

Device : cuda
GPU    : Tesla T4
Seed   : 42


In [ ]:
!pip install -q transformers peft trl bitsandbytes datasets accelerate wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.9 MB/s eta 0:00:00


In [ ]:
import transformers
import peft
import trl
import bitsandbytes
import datasets
import accelerate
import wandb

print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print(f"trl          : {trl.__version__}")
print(f"bitsandbytes : {bitsandbytes.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"accelerate   : {accelerate.__version__}")
print(f"wandb        : {wandb.__version__}")

transformers : 5.0.0
peft         : 0.19.1
trl          : 1.4.0
bitsandbytes : 0.49.2
datasets     : 4.8.5
accelerate   : 1.13.0
wandb        : 0.26.1


In [ ]:
import torch

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"Device name    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'}")
print(f"VRAM (GB)      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}" if torch.cuda.is_available() else "")
print(f"PyTorch version: {torch.__version__}")

CUDA available : True
Device name    : Tesla T4
VRAM (GB)      : 15.64
PyTorch version: 2.10.0+cu128


In [ ]:
import os
import random
import numpy as np
import torch

SEED = 42

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# Convenience: detect device once
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
print(f"Seed set to : {SEED}")

Using device: cuda
Seed set to : 42


## Load flytech/python-codes-25k and inspect

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("flytech/python-codes-25k", split="train")

print(f"Dataset size    : {len(raw_dataset)} samples")
print(f"Column names    : {raw_dataset.column_names}")
print(f"\n--- Sample 0 ---")
for col in raw_dataset.column_names:
    val = raw_dataset[0][col]
    preview = str(val)[:300]
    print(f"\n[{col}]:\n{preview}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

python-codes-25k.json:   0%|          | 0.00/26.4M [00:00<?, ?B/s]

python-codes-25k.jsonl:   0%|          | 0.00/25.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49626 [00:00<?, ? examples/s]

Dataset size    : 49626 samples
Column names    : ['output', 'instruction', 'input', 'text']

--- Sample 0 ---

[output]:
```python
tasks = []
while True:
    task = input('Enter a task or type 'done' to finish: ')
    if task == 'done': break
    tasks.append(task)
print(f'Your to-do list for today: {tasks}')
```

[instruction]:
Help me set up my daily to-do list!

[input]:
Setting up your daily to-do list...

[text]:
Help me set up my daily to-do list! Setting up your daily to-do list... ```python
tasks = []
while True:
    task = input('Enter a task or type 'done' to finish: ')
    if task == 'done': break
    tasks.append(task)
print(f'Your to-do list for today: {tasks}')
```


## Data analysis

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

df = raw_dataset.to_pandas()

print("=== Null counts ===")
print(df.isnull().sum())
print("\n=== Empty string counts ===")
print((df == "").sum())

df["text_len"] = df["text"].str.len()

print(f"\n=== Text length stats (characters) ===")
print(df["text_len"].describe().round(0))

for budget in [512, 1024, 2048]:
    pct = (df["text_len"] < budget * 4).mean() * 100
    print(f"  ~{budget} tokens budget : {pct:.1f}% of samples fit")

=== Null counts ===
output         0
instruction    0
input          0
text           0
dtype: int64

=== Empty string counts ===
output             0
instruction        0
input          42296
text               0
dtype: int64

=== Text length stats (characters) ===
count    49626.0
mean       490.0
std        318.0
min         57.0
25%        276.0
50%        396.0
75%        589.0
max       2399.0
Name: text_len, dtype: float64
  ~512 tokens budget : 99.9% of samples fit
  ~1024 tokens budget : 100.0% of samples fit
  ~2048 tokens budget : 100.0% of samples fit


In [ ]:
# 3 random samples
for i in [100, 500, 1000]:
    print(f"\n{'='*60}")
    print(f"Sample {i}")
    print(f"  INSTRUCTION : {df['instruction'][i][:120]}")
    print(f"  INPUT       : {df['input'][i][:80]}")
    print(f"  OUTPUT      : {df['output'][i][:150]}")


Sample 100
  INSTRUCTION : Remind me to take breaks during gaming
  INPUT       : Setting reminders for breaks...
  OUTPUT      : ```python
import time
break_time = int(input('How many minutes between each break? '))
while True:
    time.sleep(break_time * 60)
    print('Time to 

Sample 500
  INSTRUCTION : Open Bloomberg when the stock market opens!
  INPUT       : Stock market is opening, time for financial news...
  OUTPUT      : ```python
from datetime import datetime
import webbrowser
if datetime.now().weekday() < 5 and (datetime.now().hour == 9 and datetime.now().minute >= 3

Sample 1000
  INSTRUCTION : Download a file from a URL
  INPUT       : Downloading file from {url}...
  OUTPUT      : ```python
import requests
response = requests.get('{url}')
with open('file_name.extension', 'wb') as f:
    f.write(response.content)
```


## Load XLM-RoBERTa tokenizer

In [ ]:

from transformers import AutoTokenizer

MODEL_NAME_FFT = "FacebookAI/xlm-roberta-base"

tokenizer_fft = AutoTokenizer.from_pretrained(MODEL_NAME_FFT)

# Inspect tokenizer
print(f"Tokenizer class    : {tokenizer_fft.__class__.__name__}")
print(f"Vocab size         : {tokenizer_fft.vocab_size}")
print(f"Model max length   : {tokenizer_fft.model_max_length}")
print(f"PAD token          : {tokenizer_fft.pad_token} (id={tokenizer_fft.pad_token_id})")
print(f"EOS token          : {tokenizer_fft.eos_token} (id={tokenizer_fft.eos_token_id})")
print(f"BOS token          : {tokenizer_fft.bos_token} (id={tokenizer_fft.bos_token_id})")

# test
sample_text = df['text'][0]
tokens = tokenizer_fft(sample_text, return_tensors="pt")
print(f"\nSample tokenized length: {tokens['input_ids'].shape[1]} tokens")

Tokenizer class    : XLMRobertaTokenizer
Vocab size         : 250002
Model max length   : 512
PAD token          : <pad> (id=1)
EOS token          : </s> (id=2)
BOS token          : <s> (id=0)

Sample tokenized length: 92 tokens


## Load XLM-RoBERTa with Causal LM head

In [ ]:

from transformers import AutoModelForCausalLM, AutoConfig

# Load config
config_fft = AutoConfig.from_pretrained(MODEL_NAME_FFT)
config_fft.is_decoder = True  # required for causal LM on encoder model

# Load model with modified config
model_fft = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_FFT,
    config=config_fft,
    torch_dtype=torch.bfloat16,   
    ignore_mismatched_sizes=True


model_fft = model_fft.to(DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model_fft.parameters())
trainable_params = sum(p.numel() for p in model_fft.parameters() if p.requires_grad)

print(f"Model class          : {model_fft.__class__.__name__}")
print(f"Total parameters     : {total_params / 1e6:.1f}M")
print(f"Trainable parameters : {trainable_params / 1e6:.1f}M")
print(f"Model dtype          : {next(model_fft.parameters()).dtype}")
print(f"Model device         : {next(model_fft.parameters()).device}")
print(f"\nModel config is_decoder: {model_fft.config.is_decoder}")

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

XLMRobertaForCausalLM LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model class          : XLMRobertaForCausalLM
Total parameters     : 278.3M
Trainable parameters : 278.3M
Model dtype          : torch.bfloat16
Model device         : cuda:0

Model config is_decoder: True


## Format and tokenize dataset for FFT

In [ ]:
MAX_LENGTH_FFT = 512

def format_sample_fft(sample):
    # build a string from instructions, inputs, outputs. labels will be same as input_ids for causal LM
    instruction = sample["instruction"].strip()
    inp = sample["input"].strip()
    output = sample["output"].strip()

    if inp:
        text = f"Instruction: {instruction}\nInput: {inp}\nOutput: {output}"
    else:
        text = f"Instruction: {instruction}\nOutput: {output}"
    return {"formatted_text": text}

def tokenize_fft(sample):
    tokens = tokenizer_fft(
        sample["formatted_text"],
        truncation=True,
        max_length=MAX_LENGTH_FFT,
        padding="max_length",
    )
    # Pad positions should be ignored in loss → set to -100
    labels = [
        token_id if token_id != tokenizer_fft.pad_token_id else -100
        for token_id in tokens["input_ids"]
    ]
    tokens["labels"] = labels
    return tokens

# formatting
print("Formatting dataset...")
formatted_dataset = raw_dataset.map(format_sample_fft, batched=False)
print(f"Sample formatted text:\n{formatted_dataset[0]['formatted_text'][:300]}")

# tokenization
print("\nTokenizing dataset...")
tokenized_dataset_fft = formatted_dataset.map(
    tokenize_fft,
    batched=False,
    remove_columns=raw_dataset.column_names + ["formatted_text"]
)

print(f"\n=== Tokenized Dataset ===")
print(f"Columns  : {tokenized_dataset_fft.column_names}")
print(f"Size     : {len(tokenized_dataset_fft)}")
print(f"Sample 0 input_ids length  : {len(tokenized_dataset_fft[0]['input_ids'])}")
print(f"Sample 0 labels (first 10) : {tokenized_dataset_fft[0]['labels'][:10]}")
print(f"Sample 0 labels (-100s)    : {tokenized_dataset_fft[0]['labels'].count(-100)} padding positions masked")

Formatting dataset...


Map:   0%|          | 0/49626 [00:00<?, ? examples/s]

Sample formatted text:
Instruction: Help me set up my daily to-do list!
Input: Setting up your daily to-do list...
Output: ```python
tasks = []
while True:
    task = input('Enter a task or type 'done' to finish: ')
    if task == 'done': break
    tasks.append(task)
print(f'Your to-do list for today: {tasks}')
```

Tokenizing dataset...


Map:   0%|          | 0/49626 [00:00<?, ? examples/s]


=== Tokenized Dataset ===
Columns  : ['input_ids', 'attention_mask', 'labels']
Size     : 49626
Sample 0 input_ids length  : 512
Sample 0 labels (first 10) : [0, 72022, 10763, 12, 39527, 163, 5423, 1257, 759, 31815]
Sample 0 labels (-100s)    : 411 padding positions masked


## Use 10k subset for training

I tried using the whole dataset but it would have taken more than 12 hours to finish training.

In [ ]:
subset_fft = tokenized_dataset_fft.shuffle(seed=SEED).select(range(10_000))

split = subset_fft.train_test_split(test_size=0.05, seed=SEED)
train_dataset_fft = split["train"]
eval_dataset_fft  = split["test"]

print(f"Train samples : {len(train_dataset_fft)}")
print(f"Eval  samples : {len(eval_dataset_fft)}")

Train samples : 9500
Eval  samples : 500


## Recalculate for new dataset size

In [ ]:
TOTAL_STEPS_FFT = (len(train_dataset_fft) // 4) * 2
WARMUP_STEPS_FFT = int(0.05 * TOTAL_STEPS_FFT)

print(f"Total training steps : {TOTAL_STEPS_FFT}")
print(f"Warmup steps (5%)    : {WARMUP_STEPS_FFT}")
print(f"Estimated time       : ~{TOTAL_STEPS_FFT / 0.54 / 3600:.1f} hours")

training_args_fft = TrainingArguments(
    output_dir="./results_fft",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    num_train_epochs=2,
    learning_rate=5e-5,
    optim="adamw_torch",
    bf16=True,
    gradient_checkpointing=True,
    eval_strategy="steps",
    eval_steps=200,
    logging_steps=50,
    save_steps=200,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS_FFT,
    seed=SEED,
    report_to="none",
)

print("\nTrainingArguments reconfigured ✅")

Total training steps : 4750
Warmup steps (5%)    : 237
Estimated time       : ~2.4 hours

TrainingArguments reconfigured ✅


## Recalculate for new dataset size

In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling

data_collator_fft = DataCollatorForLanguageModeling(
    tokenizer=tokenizer_fft,
    mlm=False,
)

trainer_fft = Trainer(
    model=model_fft,
    args=training_args_fft,
    train_dataset=train_dataset_fft,
    eval_dataset=eval_dataset_fft,
    data_collator=data_collator_fft,
)

print(f"Trainer ready — {len(train_dataset_fft)} train / {len(eval_dataset_fft)} eval")
print(f"Total steps: {TOTAL_STEPS_FFT}")

Trainer ready — 9500 train / 500 eval
Total steps: 4750


## Part 1 training

In [ ]:
import time

print("Starting Part I: FFT on XLM-RoBERTa (10k subset)...")
print("="*60)

start_time = time.time()
train_result_fft = trainer_fft.train()
elapsed = time.time() - start_time

trainer_fft.save_model("./results_fft/final_model")
tokenizer_fft.save_pretrained("./results_fft/final_model")

print("\n" + "="*60)
print("TRAINING COMPLETE — Part I Summary")
print("="*60)
print(f"  Total time        : {elapsed/60:.1f} minutes")
print(f"  Final train loss  : {train_result_fft.training_loss:.4f}")
print(f"  Total steps done  : {train_result_fft.global_step}")

Starting Part I: FFT on XLM-RoBERTa (10k subset)...


Step,Training Loss,Validation Loss
200,5.660674,4.883121
400,4.069992,3.734907
600,3.735140,3.368986
800,3.334505,3.179302
1000,3.310618,3.029174
1200,3.179267,2.946005
1400,3.082730,2.870347
1600,2.990967,2.819386
1800,3.123253,2.778969
2000,2.964655,2.752355


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

google colab timed out mid training, but the training & validation losses were going down anyways

In [ ]:
part1_results = {
    "model"             : "FacebookAI/xlm-roberta-base",
    "method"            : "Full Fine-Tuning (FFT)",
    "dataset_size"      : 9500,
    "steps_completed"   : 2600,
    "total_steps"       : 4750,
    "final_train_loss"  : 2.861564,
    "final_eval_loss"   : 2.700569,
    "epochs_completed"  : 1.15,
    "batch_size"        : 4,
    "learning_rate"     : 5e-5,
    "scheduler"         : "cosine",
    "notes"             : "Training stopped at step 2600 due to Colab session crash. Loss was steadily decreasing."
}

print("="*50)
print("PART I RESULTS SUMMARY (RECORDED)")
print("="*50)
for k, v in part1_results.items():
    print(f"  {k:<25}: {v}")

print("\n✅ Part I complete — moving to Part II")

PART I RESULTS SUMMARY (RECORDED)
  model                    : FacebookAI/xlm-roberta-base
  method                   : Full Fine-Tuning (FFT)
  dataset_size             : 9500
  steps_completed          : 2600
  total_steps              : 4750
  final_train_loss         : 2.861564
  final_eval_loss          : 2.700569
  epochs_completed         : 1.15
  batch_size               : 4
  learning_rate            : 5e-05
  scheduler                : cosine
  notes                    : Training stopped at step 2600 due to Colab session crash. Loss was steadily decreasing.

✅ Part I complete — moving to Part II


# PART 2

## Load dataset and sample 2500 for Part II SFT

In [ ]:

from datasets import load_dataset

raw_dataset = load_dataset("flytech/python-codes-25k", split="train")

sft_dataset = raw_dataset.shuffle(seed=SEED).select(range(2500))

print(f"Full dataset size  : {len(raw_dataset)}")
print(f"SFT sample size    : {len(sft_dataset)}")
print(f"Columns            : {sft_dataset.column_names}")
print(f"\nSample 0 instruction: {sft_dataset[0]['instruction']}")
print(f"Sample 0 input      : {sft_dataset[0]['input'][:80]}")
print(f"Sample 0 output     : {sft_dataset[0]['output'][:150]}")

Full dataset size  : 49626
SFT sample size    : 2500
Columns            : ['output', 'instruction', 'input', 'text']

Sample 0 instruction: Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order
Sample 0 input      : 
Sample 0 output     : ```python
import random

# generating a list of unique numbers from 0 to 9 in random order
random_numbers = random.sample(range(0, 10), 10)

# sort li


## Format into Qwen chat format

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert Python programmer. "
    "Given a task description, write clean, functional, and well-structured Python code."
)

def format_sample_sft(sample):
    instruction = sample["instruction"].strip()
    inp = sample["input"].strip()
    output = sample["output"].strip()

    if inp:
        user_msg = f"{instruction}\nContext: {inp}"
    else:
        user_msg = instruction

    # Qwen chat format
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        f"<|im_start|>assistant\n{output}<|im_end|>"
    )
    return {"text": text}

sft_dataset = sft_dataset.map(format_sample_sft, remove_columns=sft_dataset.column_names)

print(f"Formatted dataset size: {len(sft_dataset)}")
print(f"Columns: {sft_dataset.column_names}")
print(f"\n--- Sample 0 formatted ---")
print(sft_dataset[0]["text"][:500])

Formatted dataset size: 2500
Columns: ['text']

--- Sample 0 formatted ---
<|im_start|>system
You are an expert Python programmer. Given a task description, write clean, functional, and well-structured Python code.<|im_end|>
<|im_start|>user
Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order<|im_end|>
<|im_start|>assistant
```python
import random

# generating a list of unique numbers from 0 to 9 in random order
random_numbers = random.sample(range(0, 10), 10)

# sort list of numbers 
random_numbers.sort()

# print sorted lis


## Solving some restarting problems that occured

In [ ]:
!pip install -q -U bitsandbytes transformers peft trl accelerate datasets

In [ ]:
import os
os.kill(os.getpid(), 9)

## Load Qwen2-1.5B-Instruct with 4-bit NF4 QLoRA

In [ ]:

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME_SFT = "Qwen/Qwen2-1.5B-Instruct"

# 4-bit NF4 quantization config 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              
    bnb_4bit_compute_dtype=torch.bfloat16,  
    bnb_4bit_use_double_quant=True,      
)

# Load tokenizer
tokenizer_sft = AutoTokenizer.from_pretrained(MODEL_NAME_SFT)
tokenizer_sft.padding_side = "right"  # required for SFT with causal LM

print(f"Tokenizer loaded:")
print(f"  Vocab size    : {tokenizer_sft.vocab_size}")
print(f"  PAD token     : {tokenizer_sft.pad_token}")
print(f"  EOS token     : {tokenizer_sft.eos_token}")
print(f"  Chat template : {'Yes' if tokenizer_sft.chat_template else 'No'}")

# Load model in 4-bit
print(f"\nLoading {MODEL_NAME_SFT} in 4-bit NF4...")
model_sft = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME_SFT,
    quantization_config=bnb_config,
    device_map="auto",
)


vram_used = torch.cuda.memory_allocated() / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"\n✅ Model loaded:")
print(f"  Class         : {model_sft.__class__.__name__}")
print(f"  VRAM used     : {vram_used:.2f} GB / {vram_total:.2f} GB")
print(f"  Total params  : {sum(p.numel() for p in model_sft.parameters())/1e6:.1f}M")

Tokenizer loaded:
  Vocab size    : 151643
  PAD token     : <|endoftext|>
  EOS token     : <|im_end|>
  Chat template : Yes

Loading Qwen/Qwen2-1.5B-Instruct in 4-bit NF4...


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


✅ Model loaded:
  Class         : Qwen2ForCausalLM
  VRAM used     : 1.15 GB / 15.64 GB
  Total params  : 888.6M


## Apply LoRA (PEFT) configuration

In [ ]:

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for k-bit training (freezes base, enables gradient checkpointing)
model_sft = prepare_model_for_kbit_training(model_sft)

# LoRA config 
lora_config = LoraConfig(
    r=16,                    
    lora_alpha=32,           
    target_modules="all-linear", 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model_sft = get_peft_model(model_sft, lora_config)

# Show trainable vs frozen parameters
trainable = sum(p.numel() for p in model_sft.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_sft.parameters())

print(f"✅ LoRA adapters attached:")
print(f"  Trainable params : {trainable/1e6:.2f}M")
print(f"  Total params     : {total/1e6:.2f}M")
print(f"  Trainable %      : {100*trainable/total:.2f}%")
print(f"  VRAM used        : {torch.cuda.memory_allocated()/1e9:.2f} GB")

✅ LoRA adapters attached:
  Trainable params : 18.46M
  Total params     : 907.08M
  Trainable %      : 2.04%
  VRAM used        : 1.69 GB


## SFT Training TRL 1.4 compatible fix

In [ ]:

from trl import SFTTrainer, SFTConfig
import time

# Set max length on tokenizer directly
tokenizer_sft.model_max_length = 1024
tokenizer_sft.truncation_side = "right"

TOTAL_STEPS_SFT = (len(sft_dataset) // (1 * 4)) * 1
WARMUP_STEPS_SFT = int(0.05 * TOTAL_STEPS_SFT)

print(f"Total SFT steps  : {TOTAL_STEPS_SFT}")
print(f"Warmup steps     : {WARMUP_STEPS_SFT}")

sft_config = SFTConfig(
    output_dir="./results_sft",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    optim="paged_adamw_8bit",

    bf16=True,
    gradient_checkpointing=True,

    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_STEPS_SFT,

    logging_steps=50,
    eval_strategy="no",
    save_steps=200,
    save_total_limit=1,

    dataset_text_field="text",

    seed=SEED,
    report_to="none",
)

trainer_sft = SFTTrainer(
    model=model_sft,
    args=sft_config,
    train_dataset=sft_dataset,
    processing_class=tokenizer_sft,
)

print("\nStarting Part II: QLoRA SFT on Qwen2-1.5B-Instruct...")
print("="*60)

start_time = time.time()
train_result_sft = trainer_sft.train()
elapsed = time.time() - start_time

trainer_sft.save_model("./results_sft/final_model")
tokenizer_sft.save_pretrained("./results_sft/final_model")

print("\n" + "="*60)
print("TRAINING COMPLETE — Part II Summary")
print("="*60)
print(f"  Total time       : {elapsed/60:.1f} minutes")
print(f"  Final train loss : {train_result_sft.training_loss:.4f}")
print(f"  Total steps      : {train_result_sft.global_step}")

Total SFT steps  : 625
Warmup steps     : 31


Adding EOS to train dataset:   0%|          | 0/2500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2500 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.



Starting Part II: QLoRA SFT on Qwen2-1.5B-Instruct...


Step,Training Loss
50,0.900478
100,0.699733
150,0.663616
200,0.685760
250,0.663142
300,0.645590
350,0.646157
400,0.630220
450,0.656996
500,0.609052



TRAINING COMPLETE — Part II Summary
  Total time       : 51.0 minutes
  Final train loss : 0.6782
  Total steps      : 625


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/NLP_Assignment4', exist_ok=True)
print("✅ Drive mounted and folder created")

Mounted at /content/drive
✅ Drive mounted and folder created


##  Save SFT model to Google Drive

In [ ]:

import shutil, os

DRIVE_SFT_PATH = "/content/drive/MyDrive/NLP_Assignment4/sft_model"

print("Copying SFT model to Google Drive...")
shutil.copytree("./results_sft/final_model", DRIVE_SFT_PATH, dirs_exist_ok=True)

# Verify
files = os.listdir(DRIVE_SFT_PATH)
print(f"✅ Saved to Drive: {DRIVE_SFT_PATH}")
print(f"   Files: {files}")

Copying SFT model to Google Drive...
✅ Saved to Drive: /content/drive/MyDrive/NLP_Assignment4/sft_model
   Files: ['training_args.bin', 'adapter_config.json', 'chat_template.jinja', 'README.md', 'tokenizer_config.json', 'adapter_model.safetensors', 'tokenizer.json']


In [ ]:
part2_results = {
    "model"             : "Qwen/Qwen2-1.5B-Instruct",
    "method"            : "QLoRA SFT (PEFT)",
    "dataset_size"      : 2500,
    "lora_rank"         : 16,
    "trainable_params"  : "18.46M (2.04%)",
    "steps_completed"   : 625,
    "num_epochs"        : 1,
    "batch_size"        : 1,
    "grad_accum"        : 4,
    "effective_batch"   : 4,
    "learning_rate"     : 2e-4,
    "scheduler"         : "cosine",
    "loss_step_50"      : 0.900478,
    "loss_step_300"     : 0.645590,
    "final_train_loss"  : 0.6782,
    "total_time_min"    : 51.0,
}

print("="*50)
print("PART II RESULTS SUMMARY")
print("="*50)
for k, v in part2_results.items():
    print(f"  {k:<25}: {v}")

print("\n✅ Part II complete — moving to Part III (DPO)")

PART II RESULTS SUMMARY
  model                    : Qwen/Qwen2-1.5B-Instruct
  method                   : QLoRA SFT (PEFT)
  dataset_size             : 2500
  lora_rank                : 16
  trainable_params         : 18.46M (2.04%)
  steps_completed          : 625
  num_epochs               : 1
  batch_size               : 1
  grad_accum               : 4
  effective_batch          : 4
  learning_rate            : 0.0002
  scheduler                : cosine
  loss_step_50             : 0.900478
  loss_step_300            : 0.64559
  final_train_loss         : 0.6782
  total_time_min           : 51.0

✅ Part II complete — moving to Part III (DPO)
